# Ablačná analýza spracovateľského reťazca

Overuje prínos jednotlivých krokov reťazca na binárnej úlohe (rest vs MI). Aby bolo porovnanie férové, mení sa vždy **jeden faktor** oproti plnému reťazcu a klasifikátor je fixný (LDA, bez Potata). Reportuje sa balanced accuracy (LOSO, úroveň trialu).

Skúmané faktory: CAR (zap/vyp), pásmová filtrácia (filter bank mu+beta vs. jediné širokopásmové filtrovanie 8–30 Hz), recentering (zap/vyp), počet kanálov (4 motorické / 8). Doplnené sú dva klasické východiská: spektrálny výkon (Welch) + LDA a CSP + LDA.

Využíva sa modul `mi_pipeline.py` (rovnaká LOSO logika ako v hlavných notebookoch).

## 1. Načítanie okien (so všetkými 8 kanálmi, voliteľný CAR)

In [6]:
import json
import numpy as np
import mne
import glob
from pathlib import Path
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.mean import mean_covariance
from pyriemann.utils.base import invsqrtm
from mne.decoding import CSP
import mi_pipeline as mi
mne.set_log_level('ERROR')

SESSION_GLOB = 'data/session_glob/brainwaves_*_T6s.json'
# mu a beta tvoria filter bank; broad = jedine sirokopasmove filtrovanie (bez filter banku)
ABL_BANDS = {'mu': (8, 12), 'beta': (13, 30), 'broad': (8, 30)}

def load_windows(path, sid, use_car):
    with open(path) as f:
        rj = json.load(f)
    ch_names = rj[0]['info']['channelNames']
    parts, labs = [], []
    for ch in rj:
        d = np.array(ch['data']); parts.append(d)
        labs += [ch.get('mark', 0)] * d.shape[1]
    sig = np.hstack(parts); labs = np.array(labs)
    info = mne.create_info(ch_names, mi.SAMPLING_RATE, 'eeg')
    raw = mne.io.RawArray(sig, info, verbose=False)
    if use_car:
        raw.set_eeg_reference('average', projection=False, verbose=False)
    diff = np.diff(labs, prepend=labs[0]); diff[0] = 1
    onsets = np.where(diff != 0)[0]; ev = labs[onsets]
    events = np.column_stack([onsets, np.zeros_like(onsets), ev])
    eid = {mi.CLASS_NAMES[k]: int(k) for k in np.unique(ev) if k in mi.CLASS_NAMES}
    win, step = int(mi.WINDOW_SEC*mi.SAMPLING_RATE), int(mi.STEP_SEC*mi.SAMPLING_RATE)
    out = {b: [] for b in ABL_BANDS}; y=g=None
    for b,(lo,hi) in ABL_BANDS.items():
        r = raw.copy().filter(lo, hi, fir_design='firwin', verbose=False)
        ep = mne.Epochs(r, events, eid, tmin=mi.EPOCH_TMIN, tmax=mi.EPOCH_TMAX,
                        baseline=None, preload=True, verbose=False)
        X = ep.get_data(copy=True); yy = ep.events[:,-1]
        ww, yl, gl = [], [], []
        for e in range(len(X)):
            for s in range(0, X.shape[-1]-win+1, step):
                ww.append(X[e,:,s:s+win]); yl.append(int(yy[e])); gl.append(f'{sid}_{e}')
        out[b]=np.array(ww); y=np.array(yl); g=np.array(gl)
    return out, y, g, ch_names

def load_all(use_car):
    files = sorted(glob.glob(SESSION_GLOB)); assert files
    per={b:[] for b in ABL_BANDS}; yl=[]; gl=[]; ch=None
    for sid,p in enumerate(files):
        w,y,g,ch = load_windows(p, sid, use_car)
        for b in ABL_BANDS: per[b].append(w[b])
        yl.append(y); gl.append(g)
    W={b:np.concatenate(v) for b,v in per.items()}
    y=np.concatenate(yl); g=np.concatenate(gl)
    sess=np.array([int(x.split('_')[0]) for x in g])
    return W, (y>0).astype(int), g, sess, ch   # binarne: rest=0, MI=1

W_car,  y, groups, sess, ch_names = load_all(use_car=True)
W_nocar, _, _, _, _               = load_all(use_car=False)
MOTOR = [ch_names.index(c) for c in mi.MOTOR_CHANNELS]
ALL8  = list(range(len(ch_names)))
print('okna:', len(y), '| rest', int((y==0).sum()), 'MI', int((y==1).sum()))

okna: 1950 | rest 636 MI 1314


## 2. Pomocné funkcie: kovariancie, (ne)recentering, LOSO s tangent space

In [7]:
def covs(windows, chan_idx):
    return Covariances(estimator=mi.COV_ESTIMATOR).transform(windows[:, chan_idx, :])

def recenter(C, sess):
    C = C.copy()
    for s in np.unique(sess):
        m = sess==s
        Ri = invsqrtm(mean_covariance(C[m], metric='riemann'))
        C[m] = Ri @ C[m] @ Ri
    return C

def make_lda():
    return make_pipeline(StandardScaler(),
        LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto', priors=[0.5,0.5]))

def covset(W, chan_idx, bands, do_recenter, sess):
    out = {}
    for b in bands:
        C = covs(W[b], chan_idx)
        out[b] = recenter(C, sess) if do_recenter else C
    return out

def loso_balacc(covs_by_band, y, groups, sess):
    """LOSO + tangent space (fit len na treningu) + LDA -> trial-level bal. acc."""
    logo = LeaveOneGroupOut(); yt_all, pt_all, g_all = [], [], []
    for tr, te in logo.split(y, y, sess):
        Ftr, Fte = [], []
        for b in covs_by_band:
            ts = TangentSpace(metric='riemann').fit(covs_by_band[b][tr])
            Ftr.append(ts.transform(covs_by_band[b][tr]))
            Fte.append(ts.transform(covs_by_band[b][te]))
        clf = make_lda(); clf.fit(np.hstack(Ftr), y[tr])
        p = clf.predict(np.hstack(Fte))
        yt_all.append(y[te]); pt_all.append(p); g_all.append(groups[te])
    yt, pt = mi._aggregate_to_trials(np.concatenate(yt_all), np.concatenate(pt_all),
                                     np.concatenate(g_all))
    return balanced_accuracy_score(yt, pt)

## 3. Ablácia kovariančného reťazca
Mení sa vždy jeden faktor oproti plnému reťazcu (CAR, 4 kanály, filter bank mu+beta, recentering). Variant **bez bandpass** používa namiesto filter banku jediné širokopásmové filtrovanie 8–30 Hz.

In [8]:
results = {}
# referencia: plny retazec (filter bank mu+beta)
results['Plný reťazec (CAR, 4 kan., filter bank, recentering)'] = loso_balacc(
    covset(W_car, MOTOR, ['mu','beta'], True, sess), y, groups, sess)
# bez CAR
results['— bez CAR'] = loso_balacc(
    covset(W_nocar, MOTOR, ['mu','beta'], True, sess), y, groups, sess)
# bez filter banku: jedine sirokopasmove filtrovanie 8-30 Hz namiesto oddelenych pasiem mu a beta
results['— bez filter banku (8–30 Hz spolu)'] = loso_balacc(
    covset(W_car, MOTOR, ['broad'], True, sess), y, groups, sess)
# bez recenteringu
results['— bez recenteringu'] = loso_balacc(
    covset(W_car, MOTOR, ['mu','beta'], False, sess), y, groups, sess)
# vsetkych 8 kanalov
results['— 8 kanálov namiesto 4'] = loso_balacc(
    covset(W_car, ALL8, ['mu','beta'], True, sess), y, groups, sess)
for k,v in results.items(): print(f'{v:.3f}  {k}')

0.799  Plný reťazec (CAR, 4 kan., filter bank, recentering)
0.800  — bez CAR
0.772  — bez filter banku (8–30 Hz spolu)
0.772  — bez recenteringu
0.860  — 8 kanálov namiesto 4


## 4. Klasické východiská (baseline)
Spektrálny výkon (Welch) + LDA a CSP + LDA, oboje LOSO na 4 motorických kanáloch.

In [9]:
from scipy.signal import welch

def psd_features(windows, chan_idx):
    """Priemerny vykon v mu a beta pasme pre kazdy kanal (log)."""
    X = windows[:, chan_idx, :]
    f, P = welch(X, fs=mi.SAMPLING_RATE, nperseg=min(256, X.shape[-1]), axis=-1)
    feats = []
    for lo,hi in [(8,12),(13,30)]:
        band = (f>=lo)&(f<=hi)
        feats.append(np.log(P[:,:,band].mean(axis=-1)+1e-12))
    return np.concatenate(feats, axis=1)

def loso_psd(W, chan_idx, y, groups, sess):
    logo=LeaveOneGroupOut(); yt_all,pt_all,g_all=[],[],[]
    F = psd_features(W['broad'], chan_idx)
    for tr,te in logo.split(y,y,sess):
        clf=make_lda(); clf.fit(F[tr], y[tr]); p=clf.predict(F[te])
        yt_all.append(y[te]); pt_all.append(p); g_all.append(groups[te])
    yt,pt=mi._aggregate_to_trials(np.concatenate(yt_all),np.concatenate(pt_all),np.concatenate(g_all))
    return balanced_accuracy_score(yt,pt)

def loso_csp(W, chan_idx, y, groups, sess, n_comp=4):
    logo=LeaveOneGroupOut(); yt_all,pt_all,g_all=[],[],[]
    X = W['broad'][:, chan_idx, :]
    for tr,te in logo.split(y,y,sess):
        csp=CSP(n_components=min(n_comp,len(chan_idx)), reg='ledoit_wolf', log=True)
        Ftr=csp.fit_transform(X[tr], y[tr]); Fte=csp.transform(X[te])
        clf=make_lda(); clf.fit(Ftr, y[tr]); p=clf.predict(Fte)
        yt_all.append(y[te]); pt_all.append(p); g_all.append(groups[te])
    yt,pt=mi._aggregate_to_trials(np.concatenate(yt_all),np.concatenate(pt_all),np.concatenate(g_all))
    return balanced_accuracy_score(yt,pt)

results['Welch (spektrálny výkon) + LDA'] = loso_psd(W_car, MOTOR, y, groups, sess)
results['CSP + LDA'] = loso_csp(W_car, MOTOR, y, groups, sess)
print(f"{results['Welch (spektrálny výkon) + LDA']:.3f}  Welch + LDA")
print(f"{results['CSP + LDA']:.3f}  CSP + LDA")

0.712  Welch + LDA
0.726  CSP + LDA


## 5. Súhrnná ablačná tabuľka

In [10]:
print(f"{'konfigurácia':54s}{'bal_acc':>9s}")
print('-'*63)
order = ['Plný reťazec (CAR, 4 kan., filter bank, recentering)',
         '— bez CAR','— bez filter banku (8–30 Hz spolu)',
         '— bez recenteringu','— 8 kanálov namiesto 4',
         'Welch (spektrálny výkon) + LDA','CSP + LDA']
for k in order:
    print(f'{k:54s}{results[k]:>9.3f}')

konfigurácia                                            bal_acc
---------------------------------------------------------------
Plný reťazec (CAR, 4 kan., filter bank, recentering)      0.799
— bez CAR                                                 0.800
— bez filter banku (8–30 Hz spolu)                        0.772
— bez recenteringu                                        0.772
— 8 kanálov namiesto 4                                    0.860
Welch (spektrálny výkon) + LDA                            0.712
CSP + LDA                                                 0.726
